# LAFUSION 2026 — Variable-agent data harvesting

This notebook produces the compact set of analyses selected for the paper. It treats the MLflow runs as experimental conditions rather than as a chronological development log.

The three research questions are:

1. Does inverse agent-count sampling correct the underrepresentation of low-agent episodes?
2. Is training-time exposure to agent deaths required for robustness to deaths at evaluation time?
3. What does the flex encoder change relative to a conventional MLP in sample efficiency, final success, and completion time?

The notebook intentionally limits the main output to four publication figures. Numerical summaries remain available as tables for checking values and writing the results section.

## Setup and reproducibility

Runs are resolved by their unique names in the `death` MLflow experiment through `data_harvesting.analysis.ExperimentRun`. Final-policy evaluations use deterministic episode seeds so the compared policies receive the same scenario sequence.

Periodic training evaluations contain 100 randomly generated episodes every 250,000 collected transitions. Their episode seeds were not fixed during training, so they are appropriate for learning-curve trends but not paired checkpoint comparisons.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from data_harvesting.analysis import ExperimentRun

TRACKING_URI = "http://localhost:5000"
EXPERIMENT_NAME = "death"
SAMPLING_EVAL_SEED = 20260819
ARCHITECTURE_EVAL_SEED = 20260827
FINAL_EVAL_EPISODES = 2_000
DEATH_EVAL_SEED = 20260823
DEATH_EVAL_EPISODES = 2_000
DEATH_PROBABILITIES = (0.0, 0.0005, 0.001, 0.002, 0.005, 0.01, 0.02)

sns.set_theme(context="paper", style="whitegrid", font_scale=1.0)
plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": True,
})

In [ ]:
RUNS = {
    name: ExperimentRun.from_name(EXPERIMENT_NAME, name, tracking_uri=TRACKING_URI)
    for name in ("3_refactor", "4_proportional", "5_deathless", "6_mlp", "7_deathful")
}

pd.DataFrame(
    [
        {"run": name, "run_id": run.run_id, "status": run.status}
        for name, run in RUNS.items()
    ]
)

In [ ]:
LABELS = {
    "3_refactor": "Uniform sampling",
    "4_proportional": "Inverse sampling",
    "5_deathless": "Flex Encoder",
    "6_mlp": "MLP",
}

# Experiment 1 — Agent-count sampling ablation

**Research question.** Does sampling the initial number of agents with probability proportional to $1/n$ correct the weaker performance observed at low agent counts?

`3_refactor` and `4_proportional` use the same medium flex architecture, training budget, hyperparameters, and training death probability. The experimental change is the initial agent-count distribution: uniform sampling in `3_refactor` and inverse sampling in `4_proportional`.

Both final policies are evaluated without deaths on the same seeded episodes. Success is grouped by the initial number of agents; sensor count remains randomized from one to 36. Seaborn estimates 95% confidence intervals from the episode-level results.

In [ ]:
sampling_episodes = pd.concat(
    [
        RUNS[run_name].final_model().evaluate(
            FINAL_EVAL_EPISODES,
            seed=SAMPLING_EVAL_SEED,
                num_workers=4,
            config_overrides={"environment": {"agent_death_probability": 0.0}},
        ).assign(condition=LABELS[run_name])
        for run_name in ("3_refactor", "4_proportional")
    ],
    ignore_index=True,
)

In [ ]:
proportional_ablation = sampling_episodes.assign(
    num_agents=lambda data: data["num_agents"].astype(int),
    success_percent=lambda data: data["all_collected"].astype(float) * 100,
)

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 3.2))
sns.barplot(data=proportional_ablation, x="num_agents", y="success_percent", hue="condition", ax=ax)

ax.set(xlabel="Initial number of agents", ylabel="Success rate (%)", ylim=(0, 103))
ax.legend(loc="lower right", title="Sampling method")

fig.tight_layout()

In [ ]:
one_agent_case = proportional_ablation.query("num_agents == 1").groupby("condition")["success_percent"].mean()
one_agent_case

In [ ]:
all_case_summary = proportional_ablation.groupby("condition")["success_percent"].agg(["mean", "std"])
all_case_summary

**Reading the figure.** The comparison is intentionally stratified by agent count because an overall average would obscure the sampling intervention. The principal evidence is the change in the one- and two-agent conditions; performance at higher counts serves as a check that the reweighting did not materially damage the already strong cases.

# Experiment 2 — Is death exposure required during training?

**Research question.** Does training with stochastic agent deaths improve robustness when agents disappear during evaluation?

`4_proportional` was trained with death probability 0.0005, whereas `5_deathless` used the same flex architecture, inverse agent-count sampling, hyperparameters, and 25M-step budget without deaths. The policies are evaluated over a common stochastic-death grid.

The comparison includes episodes with at least two initial agents. One-agent episodes are excluded because `prevent_last_agent_death` makes them deathless by construction.

In [ ]:
death_chunks = []

for probability in DEATH_PROBABILITIES:
    for run_name in ("4_proportional", "5_deathless", "7_deathful"):
        episodes = RUNS[run_name].final_model().evaluate(
            DEATH_EVAL_EPISODES,
            seed=DEATH_EVAL_SEED,
                num_workers=4,
            config_overrides={"environment": {"death_scheduler": {"type": "stochastic", "probability": probability}}},
        ).assign(
            condition={"4_proportional": "p=0.0005", "5_deathless": "p=0", "7_deathful": "p=0.002"}[run_name],
            death_probability=probability,
        )
        death_chunks.append(episodes)

death_episodes = pd.concat(death_chunks, ignore_index=True)
death_relevant = (
    death_episodes.query("num_agents > 1")
    .assign(success_percent=lambda data: data["all_collected"].astype(float) * 100)
)

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.0))
sns.barplot(
    data=death_relevant,
    x="death_probability",
    y="success_percent",
    hue="condition",
    ax=ax,
    hue_order=["p=0", "p=0.0005", "p=0.002"]
)

ax.set(xlabel="Agent death probability", ylabel="Success rate (%)", ylim=(0, 103))
ax.legend(loc="lower right")

fig.tight_layout()
plt.show()

**Reading the figure.** Overlapping bars indicate that explicit death exposure during training did not produce a detectable robustness advantage under these conditions. The probability controls stochastic exposure rather than guaranteeing that a death occurs in every episode, and longer episodes accumulate more opportunities for death.

# Experiment 3 — Flex encoder versus conventional MLP

**Research question.** What does the flex encoder change relative to a conventional MLP when both are trained across the same variable agent-count distribution without deaths?

`5_deathless` uses the medium flex encoder; `6_mlp` disables it and uses the conventional actor and critic networks. Both use inverse agent-count sampling and a 25M-step training budget. This experiment separates two outcomes: how quickly a useful policy is learned and how the final policies behave as the number of agents changes.

## 3A. Training sample efficiency

The periodic `eval/all_collected` histories are plotted against collected environment transitions. Raw 100-episode evaluations remain visible; a centered three-evaluation rolling mean emphasizes the learning trend without concealing instability. This measures sample efficiency, not wall-clock efficiency.

In [ ]:
learning_curves = []

for run_name in ("5_deathless", "6_mlp"):
    history = RUNS[run_name].metrics("eval/all_collected").rename(
        columns={"eval/all_collected": "success_rate"}
    )
    history["condition"] = LABELS[run_name]
    history["steps_millions"] = history["step"] / 1_000_000
    history["success_percent"] = 100 * history["success_rate"]
    history["rolling_success_percent"] = history["success_percent"].rolling(3, center=True, min_periods=1).mean()
    learning_curves.append(history)
learning_curves = pd.concat(learning_curves, ignore_index=True)
learning_curves.head()

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 3.2))
sns.lineplot(
    data=learning_curves, x="steps_millions", y="success_percent",
    hue="condition", hue_order=["Flex Encoder", "MLP"],
    estimator=None, alpha=0.22, errorbar=("ci", 95), linewidth=1, legend=False, ax=ax,
)
sns.lineplot(
    data=learning_curves, x="steps_millions", y="rolling_success_percent",
    hue="condition", hue_order=["Flex Encoder", "MLP"],
    estimator=None, linewidth=2.2, ax=ax,
)
ax.set(xlabel="Collected training steps (millions)", ylabel="Periodic evaluation success (%)", xlim=(0, 25), ylim=(0, 103))
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

In [ ]:
def first_sustained_step(frame, threshold, evaluations=3):
    values = frame.sort_values("step")["success_rate"].to_numpy()
    steps = frame.sort_values("step")["step"].to_numpy()
    for index in range(len(values) - evaluations + 1):
        if np.all(values[index:index + evaluations] >= threshold):
            return steps[index]
    return np.nan

milestones = pd.DataFrame(
    [
        {
            "condition": condition,
            "threshold": threshold,
            "sustained_step": first_sustained_step(
                learning_curves.query("condition == @condition"), threshold
            ),
        }
        for condition in ("Flex Encoder", "MLP")
        for threshold in (0.50, 0.80, 0.90, 0.95)
    ]
)
milestones["steps_millions"] = milestones["sustained_step"] / 1_000_000
milestones.pivot(index="threshold", columns="condition", values="steps_millions")

**Reading the figure.** Earlier threshold crossings and a larger area under the success curve indicate greater sample efficiency. The comparison uses one training run per architecture, so it establishes the behavior of these trained policies but does not estimate training-seed variance.

## 3B. Final performance as agent count changes

The final policies are evaluated without deaths on the same seeded episodes. The first figure reports success with seaborn's 95% confidence intervals. The second reports mean completion time only for paired episodes completed successfully by both policies, preventing failed episodes—whose completion time is set to the maximum—from distorting the coordination-speed comparison. The figures are kept separate so they can be composed as subfigures in LaTeX.

In [ ]:
architecture_chunks = []
architecture_overrides = {
    "5_deathless": {"environment": {"agent_death_probability": 0.0}},
    "6_mlp": {"environment": {"death_scheduler": {"type": "stochastic", "probability": 0.0}}},
}
for run_name in ("5_deathless", "6_mlp"):
    architecture_chunks.append(
        RUNS[run_name].final_model().evaluate(
            FINAL_EVAL_EPISODES,
            seed=ARCHITECTURE_EVAL_SEED,
                num_workers=4,
            config_overrides=architecture_overrides[run_name],
        ).assign(condition=LABELS[run_name])
    )
architecture_episodes = pd.concat(architecture_chunks, ignore_index=True).assign(
    num_agents=lambda data: data["num_agents"].astype(int),
    success_percent=lambda data: data["all_collected"].astype(float) * 100,
)

In [ ]:
flex_episodes = architecture_episodes.query("condition == 'Flex Encoder'")[
    ["run_index", "num_agents", "num_sensors", "all_collected", "completion_time"]
].rename(columns={"all_collected": "flex_success", "completion_time": "flex_time"})
mlp_episodes = architecture_episodes.query("condition == 'MLP'")[
    ["run_index", "num_agents", "num_sensors", "all_collected", "completion_time"]
].rename(columns={"num_agents": "mlp_agents", "num_sensors": "mlp_sensors", "all_collected": "mlp_success", "completion_time": "mlp_time"})
paired = flex_episodes.merge(mlp_episodes, on="run_index", validate="one_to_one")
assert (paired["num_agents"] == paired["mlp_agents"]).all()
assert (paired["num_sensors"] == paired["mlp_sensors"]).all()
paired_successes = paired.query("flex_success == 1 and mlp_success == 1")
completion_episodes = paired_successes.melt(
    id_vars=["run_index", "num_agents"],
    value_vars=["flex_time", "mlp_time"],
    var_name="condition",
    value_name="completion_time",
).replace({"condition": {"flex_time": "Flex Encoder", "mlp_time": "MLP"}})

In [ ]:
fig, ax = plt.subplots(figsize=(4.0, 3.0))
sns.barplot(
    data=architecture_episodes, x="num_agents", y="success_percent",
    hue="condition", hue_order=["Flex Encoder", "MLP"], ax=ax, errorbar=("ci", 95)
)
ax.set(xlabel="Initial number of agents", ylabel="Success rate (%)", ylim=(0, 103))
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4.0, 3.0))
sns.barplot(
    data=completion_episodes, x="num_agents", y="completion_time",
    hue="condition", hue_order=["Flex Encoder", "MLP"],
    errorbar=("ci", 95), ax=ax,
)
ax.set(xlabel="Initial number of agents", ylabel="Completion time (s)", xticks=range(1, 9))
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

**Reading the figures.** Final success is near ceiling for both policies, with the MLP's clearest advantage in the single-agent case. Completion time exposes a different architectural effect: the flex policy completes common successful scenarios increasingly faster as the number of cooperating agents grows. Together with the learning curve, this supports a sample-efficiency and multi-agent coordination-efficiency claim rather than a claim that only flex encoding can tolerate deaths.

# Reporting note

Before freezing the paper, record the repository commit, MLflow run IDs, evaluation seeds, episode counts, and whether additional independent training seeds were run. Repeated evaluation episodes quantify uncertainty of a fixed policy; they do not replace independent training seeds when making claims about training variability.